# Validación de la ingesta reproducible

Ejecuta el punto 10 desde cero para el primer mes configurado: descubre enlaces oficiales, descarga viajes y nomenclatura, normaliza/limpia y escribe Parquet y manifiesto. Todo se realiza en un directorio temporal y se elimina al terminar; no se reemplazan datos ni manifiestos compartidos. Solo se muestran conteos y versión, nunca viajes individuales. Esta prueba real de un mes complementa las pruebas automatizadas con datos sintéticos; no certifica una ejecución completa de los 36 meses.

In [1]:
from pathlib import Path
import sys
import tempfile

root = Path.cwd().resolve()
while not (root / 'proyecto_mibici').is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from proyecto_mibici import config
from proyecto_mibici.dataset import DataManifest, ejecutar_ingesta, obtener_version_datos

with tempfile.TemporaryDirectory(prefix='mibici-validacion-') as directory:
    temporary = Path(directory)
    manifest_path = temporary / 'data_manifest.json'
    manifest = ejecutar_ingesta(
        anio_inicio=config.DEFAULT_ANIO_INICIO, mes_inicio=config.DEFAULT_MES_INICIO,
        anio_fin=config.DEFAULT_ANIO_INICIO, mes_fin=config.DEFAULT_MES_INICIO,
        input_path=temporary / 'raw', parquet_path=temporary / 'viajes.parquet',
        estaciones_path=temporary / 'estaciones.parquet', manifiesto_path=manifest_path,
    )
    errors = DataManifest(manifest_path).verificar()
    assert not errors, errors
    assert manifest['excluye_datos_personales']
    assert len(list((temporary / 'raw').glob('*.csv'))) == 2
    print('Periodo validado:', manifest['rango_fechas'])
    print('Resumen:', manifest['resumen'])
    print('Versión de datos:', obtener_version_datos(manifest_path))
    print('Columnas comunes:', manifest['salida']['columnas'])
    print('Verificación de hashes, filas y esquemas: correcta')
print('Archivos temporales eliminados; datos compartidos sin cambios.')


2026-10-20 23:47:34.092 | INFO     | proyecto_mibici.dataset:ejecutar_ingesta:715 - 2023-10: 383239 registros válidos
Periodo validado: {'inicio': '2023-10', 'fin': '2023-10'}
Resumen: {'n_meses': 1, 'n_viajes': 383239, 'n_estaciones_nomenclatura': 484, 'n_viajes_limpios': 383239}
Versión de datos: 459f2e0b9973
Columnas comunes: ['viaje_id', 'inicio_viaje', 'fin_viaje', 'origen_id', 'destino_id', 'fecha_archivo']
Verificación de hashes, filas y esquemas: correcta
Archivos temporales eliminados; datos compartidos sin cambios.
